<a href="https://colab.research.google.com/github/Jayalekshmi872/ict-aiml-internship/blob/main/demo_intern_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#configuration

import re, math, time, json
import pandas as pd
from IPython.display import HTML, display

EXCEL_PATH   = 'video_metadata.xlsx' #excel path
BUFFER_SEC   = 25      #how many seconds of video to include before and after a detected moment.
MIN_LEN 30             #30-60 sec
MAX_LEN = 60
ANSWER_WINDOW_SEC = 20 # an answer must start within 20 seconds after the question
USE_ZERO_SHOT = True   #pretrained zero-shot classification model # False = cue-phrase only (fast, no model download)
ZS_THRESHOLD  = 0.55   # min probability to accept agree/disagree/Q-A
TOP_K_PER_TYPE = 8     # max segments per vedio

In [ ]:
#extracts the unique YouTube video ID

def extract_video_id(url):
    m = re.search(r'(?:youtu\.be/|v=|/embed/|/shorts/)([A-Za-z0-9_-]{11})', str(url))
    return m.group(1) if m else None #return ID else None

videos = pd.read_excel(EXCEL_PATH)
videos['yt_id'] = videos['video_url'].apply(extract_video_id)
print(videos[['video_id','video_title','yt_id']])
assert videos.yt_id.notna().all(), 'Some URLs could not be parsed'

  video_id                                    video_title        yt_id
0     V001                         confidence isn't  loud  rKpltaOMFdc
1     V002                       Does money buy happiness  bdpyQm5l78o
2     V003                 You cannot always be positive   -pYOlucclio
3     V004                                   We the women  63mazZL-Lh4
4     V005  The future mark zuckerberg is trying to build  oX7OduG1YmI


In [ ]:
!pip install -q youtube-transcript-api #install library

en – English

en-US – American English

en-GB – British English

en-IN – Indian English

In [ ]:
#collect trancript with timestamps

from youtube_transcript_api import YouTubeTranscriptApi

def fetch_transcript(yt_id, langs=('en','en-US','en-GB','en-IN')):
    """Returns list of {'text','start','duration'} or None."""
    try:
        try:                                    # v1.x API
            api = YouTubeTranscriptApi()
            try:
                fetched = api.fetch(yt_id, languages=list(langs)) #new version
            except Exception:                   # no English track- take whatever exists
                tr = next(iter(api.list(yt_id)))
                fetched = tr.fetch()
            return [{'text': s.text, 'start': s.start, 'duration': s.duration} for s in fetched]
        except AttributeError:                  # v0.x API
            return YouTubeTranscriptApi.get_transcript(yt_id, languages=list(langs)) #old version
    except Exception as e:
        print(f'  transcript failed for {yt_id}: {type(e).__name__}: {str(e)[:120]}')
        return None

In [ ]:
#convert small YouTube transcript fragments into more meaningful sentences using word-level timestamps.


'''Converts transcript fragments into individual words
   and estimates their timestamps'''
   #help in overlaping timestamp
def to_words(snips):
    words = []  #store individual words
    for i, s in enumerate(snips):
        txt = re.sub(r'\[[^\]]*\]|\([^)]*\)', ' ', s['text']).replace('\n', ' ')   # drop [Music], (laughs)
        #removes unwanted descriptions [Music], (laughs) from the transcript.replace with space
        #re.sub-replace a pattern
        ws = txt.split() #split nto individual words.
        if not ws: continue #whitespace skip eg music
        start = s['start']
        dur = s['duration']
        #handle overlaping caption
        if i + 1 < len(snips):
            dur = min(dur, max(snips[i+1]['start'] - start, 0.2))
        dur = max(dur, 0.2)
        for k, w in enumerate(ws):
            words.append({
                'w': w,  #word
                't': start + dur*k/len(ws), #estimted start time
                'e': start + dur*(k+1)/len(ws)  #estimted end time
                })
    return words

'''Groups those words into larger text segments that
     resemble sentences.'''
def build_sentences(snips, max_words=30, pause=1.2): #max word in group 30,pause of more than 1.2 seconds can indicate a new group
    words = to_words(snips)
    if not words: return [], False
    punct_ratio = sum(w['w'][-1] in '.?!' for w in words) / len(words) #whether the words have punctuation at their endings
    has_punct = punct_ratio > 0.01 #Why?-it can help identify sentence boundaries.
    sents, cur, turn = [], [], False
    #sents: Stores the completed sentence groups.
    #cur:   Stores the words in the current group.
    #turn:  Tracks whether a speaker-change marker was detected.
    def flush(): #Saves the collected words as a group
        nonlocal cur, turn
        if cur:
            sents.append({'text': ' '.join(x['w'] for x in cur),
                          'start': cur[0]['t'],
                          'end': cur[-1]['e'],
                          'n': len(cur),
                          'turn': turn})
        cur, turn = [], False
    for w in words:
        tok = w['w']
        #to identify when a new speaker starts talking.
        if tok.startswith('>>'):                                  # speaker change marker
            flush(); turn = True
            tok = tok.lstrip('>').strip()
            if not tok: continue
            w = {**w, 'w': tok}
        elif cur:
            gap = w['t'] - cur[-1]['e'] #previous word - current word
            prev_end = cur[-1]['w'][-1] in '.?!'#previous word ended with punctuation.
            if prev_end or (
                not has_punct and
                (gap > pause or len(cur) >= max_words)
                ) or len(cur) >= 45:
                flush()
        cur.append(w)
    flush()
    return sents, has_punct


The first function, `to_words()`, cleans unnecessary descriptions such as music and laughter, separates transcript snippets into individual words, and assigns estimated start and end timestamps to each word. It also handles overlapping caption durations.

The second function, `build_sentences()`, groups the individual words into meaningful text segments. It uses punctuation, time gaps, word limits, and available speaker-change markers to identify possible sentence boundaries.

The purpose of this stage is to convert small transcript fragments into more readable text segments that can be used in the subsequent NLP classification stage.

The generated word timestamps are estimated from caption-level timestamps rather than obtained directly from word-level speech recognition.


The code creates a new group when:

The previous word ends with punctuation.

Punctuation is unavailable and the time gap exceeds 1.2 seconds.

Punctuation is unavailable and the current group has at least 30 words.

The current group reaches 45 words, regardless of punctuation.

In [ ]:
#Question–Answer detection

#Question words
WH  = r'(what|why|how|when|where|who|whom|whose|which)'
#Helping verbs
AUX = r'(do|does|did|is|are|was|were|can|could|would|will|should|have|has|had|am|shall|may|might|isn\'t|aren\'t|don\'t|doesn\'t|didn\'t|won\'t)'
#Identifying possible question beginnings
Q_START = re.compile(rf'^(so |and |but |okay |ok |well )?({WH}|{AUX})\b', re.I)#case-insensitive.

#function checks whether a sentence is a question.
def is_question(s, has_punct):
    t = s['text'].strip()
    if '?' in t: return True #? undo
    if has_punct: return False      # punctuated captions: '?'
    return bool(Q_START.match(t)) and 3 <= s['n'] <= 28
#Does the sentence begin with a question word or helping verb?
#Does it contain between 3 and 28 words?

#find an answer after identifying a question.
def find_qa(sents, has_punct):
    out = []   #out is an empty list where detected
    for i, s in enumerate(sents):
      #sentence is not identified as a question, it skips it and moves to the next one.
        if not is_question(s, has_punct): continue
        ans, j, words = [], i + 1, 0
        #ans: stores possible answer sentences.
        #j: points to the sentence immediately after the question.
       #words: counts the words in the answer.
        while j < len(sents) and len(ans) < 3:
          #t will collect a maximum of 3 answer sentences.
            nx = sents[j]
            if nx['start'] - s['end'] > (ANSWER_WINDOW_SEC if not ans else 4): break
            if is_question(nx, has_punct): break
            ans.append(nx); words += nx['n']; j += 1
        if words < 4: continue                  # no real answer follows
        conf = 0.6 + 0.2*('?' in s['text']) + 0.2*ans[0]['turn']   # +speaker-change evidence
        out.append({'type': 'Question-Answer', 'm_start': s['start'], 'm_end': ans[-1]['end'],
                    'conf': round(min(conf, 1.0), 2), 'text': s['text'] + '  ->  ' + ' '.join(a['text'] for a in ans)})
    return out

In [ ]:
AGREE_CUES = [r"i agree", r"i totally agree", r"couldn'?t agree more", r"exactly", r"absolutely", r"you'?re (absolutely )?right",
              r"you are right", r"that'?s (so )?(true|right|correct)", r"definitely", r"for sure", r"well said", r"fair (point|enough)",
              r"i think so too", r"same (here|for me)", r"100 ?%", r"makes sense", r"^yes\b", r"^yeah\b", r"^true\b", r"i feel the same"]
DISAGREE_CUES = [r"i disagree", r"i don'?t agree", r"i don'?t think so", r"not really", r"that'?s not (true|right|correct|it)",
                 r"i wouldn'?t say", r"i'?m not sure about that", r"on the contrary", r"i beg to differ", r"not necessarily",
                 r"i don'?t buy", r"that'?s wrong", r"^no\b", r"^nope\b", r"^but\b", r"^actually\b", r"^however\b", r"hold on", r"i('d| would) push back", r"i don'?t think"]
A_RE = re.compile('|'.join(AGREE_CUES), re.I)
D_RE = re.compile('|'.join(DISAGREE_CUES), re.I)

def candidates(sents):
    c = []
    for i in range(1, len(sents)):
        head = ' '.join(sents[i]['text'].split()[:14])         # reactions usually open the reply
        a, d = bool(A_RE.search(head)), bool(D_RE.search(head))
        if (a or d) and sents[i]['n'] >= 2 and sents[i]['start'] - sents[i-1]['end'] < 8:
            c.append((i, 'agree' if a and not d else 'disagree' if d and not a else 'both'))
    return c

_clf = None
def zs_classify(pairs):
    global _clf
    if _clf is None:
        import torch
        from transformers import pipeline
        _clf = pipeline('zero-shot-classification', model='facebook/bart-large-mnli',
                        device=0 if torch.cuda.is_available() else -1)
    labels = ['agrees', 'disagrees', 'neither agrees nor disagrees']
    res = _clf(pairs, candidate_labels=labels,
               hypothesis_template='In this exchange, speaker 2 {} with speaker 1.', batch_size=8)
    if isinstance(res, dict): res = [res]
    return [(r['labels'][0], r['scores'][0]) for r in res]

def find_agree_disagree(sents):
    cands = candidates(sents)
    if not cands: return []
    out = []
    if USE_ZERO_SHOT:
        pairs = [f"Speaker 1: {sents[i-1]['text']} Speaker 2: {sents[i]['text']}" for i, _ in cands]
        preds = zs_classify(pairs)
    else:
        preds = [({'agree': 'agrees', 'disagree': 'disagrees'}.get(k, 'neither agrees nor disagrees'), 0.7) for _, k in cands]
    for (i, _), (lab, sc) in zip(cands, preds):
        if lab.startswith('neither') or sc < ZS_THRESHOLD: continue
        out.append({'type': 'Agreement' if lab == 'agrees' else 'Disagreement',
                    'm_start': sents[i-1]['start'], 'm_end': sents[i]['end'], 'conf': round(float(sc), 2),
                    'text': sents[i-1]['text'] + '  ->  ' + sents[i]['text']})
    return out

In [ ]:
def make_segment(m_start, m_end, y, video_len):
    x = (m_start + m_end) / 2
    half = max(y, (m_end - m_start) / 2 + 2)
    half = min(max(half, MIN_LEN / 2), MAX_LEN / 2)
    s, e = x - half, x + half
    if s < 0:                       e = e - s; s = 0.0
    if video_len and e > video_len: s = max(0.0, s - (e - video_len)); e = video_len
    return s, e, x

def overlap_frac(a, b):
    inter = max(0, min(a[1], b[1]) - max(a[0], b[0]))
    return inter / max(1e-9, min(a[1]-a[0], b[1]-b[0]))

def select_segments(moments, y, video_len):
    segs = []
    for m in moments:
        s, e, x = make_segment(m['m_start'], m['m_end'], y, video_len)
        segs.append({**m, 'x': x, 'seg_start': s, 'seg_end': e})
    segs.sort(key=lambda r: -r['conf'])
    kept = []
    for r in segs:                                             # non-max suppression within the same type
        if any(k['type'] == r['type'] and overlap_frac((k['seg_start'], k['seg_end']), (r['seg_start'], r['seg_end'])) > 0.5 for k in kept):
            continue
        if sum(k['type'] == r['type'] for k in kept) >= TOP_K_PER_TYPE: continue
        kept.append(r)
    return sorted(kept, key=lambda r: r['seg_start'])

def fmt(t): t = int(t); return f'{t//3600:02d}:{(t%3600)//60:02d}:{t%60:02d}'

In [ ]:
def process_video(row, y=BUFFER_SEC):
    snips = fetch_transcript(row.yt_id)
    if not snips: return pd.DataFrame()
    sents, has_punct = build_sentences(snips)
    video_len = snips[-1]['start'] + snips[-1]['duration']
    print(f"{row.video_id} | {row.video_title[:40]:40s} | {len(snips)} fragments -> {len(sents)} sentences | punctuated={has_punct}")
    moments = find_qa(sents, has_punct) + find_agree_disagree(sents)
    segs = select_segments(moments, y, video_len)
    rows = []
    for r in segs:
        in_seg = ' '.join(s['text'] for s in sents if s['start'] >= r['seg_start'] and s['end'] <= r['seg_end'])
        rows.append({'video_id': row.video_id, 'title': row.video_title, 'yt_id': row.yt_id, 'type': r['type'],
                     'moment_time': fmt(r['x']), 'seg_start': round(r['seg_start'], 1), 'seg_end': round(r['seg_end'], 1),
                     'duration_s': round(r['seg_end'] - r['seg_start'], 1), 'confidence': r['conf'],
                     'key_exchange': r['text'][:300], 'segment_transcript': in_seg,
                     'watch_url': f"https://www.youtube.com/watch?v={row.yt_id}&t={int(r['seg_start'])}s",
                     'embed_url': f"https://www.youtube.com/embed/{row.yt_id}?start={int(r['seg_start'])}&end={int(r['seg_end'])}"})
    return pd.DataFrame(rows)

frames = [process_video(r) for r in videos.itertuples()]
results = pd.concat([f for f in frames if len(f)], ignore_index=True)
results.to_csv('extracted_moments.csv', index=False)
print(results.groupby(['video_id', 'type']).size().unstack(fill_value=0))
results[['video_id','type','moment_time','seg_start','seg_end','duration_s','confidence','key_exchange','watch_url']]

V001 | confidence isn't  loud                   | 1412 fragments -> 361 sentences | punctuated=False


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

V002 | Does money buy happiness                 | 1104 fragments -> 287 sentences | punctuated=False
V003 | You cannot always be positive            | 1136 fragments -> 276 sentences | punctuated=False
V004 | We the women                             | 697 fragments -> 387 sentences | punctuated=True
V005 | The future mark zuckerberg is trying to  | 477 fragments -> 326 sentences | punctuated=True
type      Agreement  Disagreement  Question-Answer
video_id                                          
V001              3             8                0
V002              2             8                0
V003              4             8                0
V004              1             5                8
V005              4             8                8


,video_id,type,moment_time,seg_start,seg_end,duration_s,confidence,key_exchange,watch_url
0,V001,Disagreement,00:08:36,491.9,541.9,50.0,0.95,just keep going like the worst that can happen...,https://www.youtube.com/watch?v=rKpltaOMFdc&t=...
1,V001,Agreement,00:11:49,684.6,734.6,50.0,0.99,and we consume so much stuff that just sounds ...,https://www.youtube.com/watch?v=rKpltaOMFdc&t=...
2,V001,Disagreement,00:15:23,898.7,948.7,50.0,0.99,you're you know dealing with potentially not f...,https://www.youtube.com/watch?v=rKpltaOMFdc&t=...
3,V001,Disagreement,00:19:58,1173.2,1223.2,50.0,0.98,am I for that person or am I not and usually I...,https://www.youtube.com/watch?v=rKpltaOMFdc&t=...
4,V001,Disagreement,00:21:08,1243.2,1293.2,50.0,0.95,voice in learning in communicating in understa...,https://www.youtube.com/watch?v=rKpltaOMFdc&t=...
...,...,...,...,...,...,...,...,...,...
62,V005,Question-Answer,00:26:42,1577.2,1627.2,50.0,0.80,and is important to preserve like building a m...,https://www.youtube.com/watch?v=oX7OduG1YmI&t=...
63,V005,Disagreement,00:28:04,1659.6,1709.6,50.0,0.99,iterative feedback loop around like okay like ...,https://www.youtube.com/watch?v=oX7OduG1YmI&t=...
64,V005,Disagreement,00:35:58,2133.7,2183.7,50.0,0.92,I feel some specific concerns around the way t...,https://www.youtube.com/watch?v=oX7OduG1YmI&t=...
65,V005,Disagreement,00:38:07,2263.0,2313.0,50.0,0.89,learn new skills now or just think about this ...,https://www.youtube.com/watch?v=oX7OduG1YmI&t=...


In [ ]:
def preview(i):
    r = results.iloc[i]
    print(f"{r.type} | {r.title} | {r.seg_start}s - {r.seg_end}s | conf {r.confidence}\n{r.key_exchange}")
    display(HTML(f'<iframe width="560" height="315" src="{r.embed_url}" frameborder="0" allowfullscreen></iframe>'))
preview(0)

In [ ]:
results

,video_id,title,yt_id,type,moment_time,seg_start,seg_end,duration_s,confidence,key_exchange,segment_transcript,watch_url,embed_url
0,V001,confidence isn't loud,rKpltaOMFdc,Disagreement,00:08:36,491.9,541.9,50.0,0.95,just keep going like the worst that can happen...,something's just come up and this was like 6 t...,https://www.youtube.com/watch?v=rKpltaOMFdc&t=...,https://www.youtube.com/embed/rKpltaOMFdc?star...
1,V001,confidence isn't loud,rKpltaOMFdc,Agreement,00:11:49,684.6,734.6,50.0,0.99,and we consume so much stuff that just sounds ...,unemployed that I was telling you back in the ...,https://www.youtube.com/watch?v=rKpltaOMFdc&t=...,https://www.youtube.com/embed/rKpltaOMFdc?star...
2,V001,confidence isn't loud,rKpltaOMFdc,Disagreement,00:15:23,898.7,948.7,50.0,0.99,you're you know dealing with potentially not f...,question because the reality is it's uncomfort...,https://www.youtube.com/watch?v=rKpltaOMFdc&t=...,https://www.youtube.com/embed/rKpltaOMFdc?star...
3,V001,confidence isn't loud,rKpltaOMFdc,Disagreement,00:19:58,1173.2,1223.2,50.0,0.98,am I for that person or am I not and usually I...,studied this even when it comes to like commun...,https://www.youtube.com/watch?v=rKpltaOMFdc&t=...,https://www.youtube.com/embed/rKpltaOMFdc?star...
4,V001,confidence isn't loud,rKpltaOMFdc,Disagreement,00:21:08,1243.2,1293.2,50.0,0.95,voice in learning in communicating in understa...,you're doing now that's something you're helpi...,https://www.youtube.com/watch?v=rKpltaOMFdc&t=...,https://www.youtube.com/embed/rKpltaOMFdc?star...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,V005,The future mark zuckerberg is trying to build,oX7OduG1YmI,Question-Answer,00:26:42,1577.2,1627.2,50.0,0.80,and is important to preserve like building a m...,and is important to preserve like building a m...,https://www.youtube.com/watch?v=oX7OduG1YmI&t=...,https://www.youtube.com/embed/oX7OduG1YmI?star...
63,V005,The future mark zuckerberg is trying to build,oX7OduG1YmI,Disagreement,00:28:04,1659.6,1709.6,50.0,0.99,iterative feedback loop around like okay like ...,iterative feedback loop around like okay like ...,https://www.youtube.com/watch?v=oX7OduG1YmI&t=...,https://www.youtube.com/embed/oX7OduG1YmI?star...
64,V005,The future mark zuckerberg is trying to build,oX7OduG1YmI,Disagreement,00:35:58,2133.7,2183.7,50.0,0.92,I feel some specific concerns around the way t...,I feel both like I really want to be optimisti...,https://www.youtube.com/watch?v=oX7OduG1YmI&t=...,https://www.youtube.com/embed/oX7OduG1YmI?star...
65,V005,The future mark zuckerberg is trying to build,oX7OduG1YmI,Disagreement,00:38:07,2263.0,2313.0,50.0,0.89,learn new skills now or just think about this ...,learn new skills now or just think about this ...,https://www.youtube.com/watch?v=oX7OduG1YmI&t=...,https://www.youtube.com/embed/oX7OduG1YmI?star...
